In [ ]:
import pandas as pd
#from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
import json
import requests
#from pymongo import MongoClient
#import pymysql
from datetime import datetime,timedelta
#import matplotlib.pyplot as plt
import psycopg2
import ast

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 5000)
pd.set_option('display.width', 100000)



In [ ]:
txn=pd.read_sql('''select 
json_extract_path_text(extra_data, 'plazaCode') as plaza_code,
json_extract_path_text(extra_data, 'laneDirection') as lane_direction
from fastag_transaction where created_at > 'Jan 1, 2023, 2:20 PM' and category='TOLL_DEBIT' 
''',con)

In [ ]:
txn.lane_direction.value_counts().keys()

In [ ]:
txn=txn[~txn['lane_direction'].isin(['', 'NA', ' '])]

In [ ]:
a=pd.DataFrame(txn.groupby('plaza_code').lane_direction.value_counts())

In [ ]:
a.reset_index(level=0, inplace=True)


In [ ]:
plaza_code_list=a.plaza_code.value_counts().keys()

In [ ]:
plaza_code_count_lane=a.plaza_code.value_counts().values

In [ ]:
plaz=pd.DataFrame()

In [ ]:
plaz['plaza_code']=plaza_code_list

In [ ]:
plaz['plaza_code_lane_count']=plaza_code_count_lane

In [ ]:
plaza_code_with_one_lane=plaz[plaz['plaza_code_lane_count']<2]['plaza_code']

In [ ]:
a=a[a['plaza_code'].isin(plaza_code_with_one_lane)].rename(columns={'lane_direction':'count_of_txn'})

In [ ]:
len(a[a['count_of_txn']>200])

In [ ]:
len(a[a['count_of_txn']<50])

In [ ]:
len(plaza_code_with_one_lane.to_list())

In [ ]:
a.to